In [28]:
from pathlib import Path
import pandas as pd
import numpy as np

current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

pickle_folder_path = project_root / "data" / "processed"

crop_yield_df1 = pd.read_pickle(pickle_folder_path / "crop_yield1.pkl")
crop_yield_df2 = pd.read_pickle(pickle_folder_path / "crop_yield2.pkl")
crop_yield_df = pd.concat([crop_yield_df1, crop_yield_df2], ignore_index=True)
geography_df = pd.read_pickle(pickle_folder_path / "geography.pkl")
precipitation1_df = pd.read_pickle(pickle_folder_path / "precipitation1.pkl")
precipitation2_df = pd.read_pickle(pickle_folder_path / "precipitation2.pkl")
precipitation3_df = pd.read_pickle(pickle_folder_path / "precipitation3.pkl")
precipitation_df = pd.concat(
    [precipitation1_df, precipitation2_df, precipitation3_df], ignore_index=True
)
vegetation_df = pd.read_pickle(pickle_folder_path / "vegetation.pkl")

In [29]:
crop_yield_cols = ["year", "state", "district", "crop_name", "crop_type", "season"]
cleaned_crop_yield_df = crop_yield_df.groupby(
    crop_yield_cols, as_index=False, observed=True
).agg(
    {
        "area": "mean",
        "production": "mean",
        "yield": "mean",
        "area_unit": "first",
        "production_unit": "first",
        "yield_unit": "first",
    }
)

In [30]:
season_months = {
    "Kharif": (6, 10),  # June to October (monsoon)
    "Rabi": (11, 3),  # November to March (winter) - wraps around year
    "Zaid": (3, 6),  # March to June (summer)
    "Summer": (3, 6),  # Same as Zaid
    "Autumn": (9, 11),  # September to November
    "Winter": (12, 2),  # December to February
    "Whole Year": (1, 12),  # All months
    "Total": (1, 12),  # Aggregate - all months
}


def add_month(df):
    df = df.copy()
    df["season_start_month"] = df["season"].map(
        lambda s: season_months.get(s, (None, None))[0]
    )
    df["season_end_month"] = df["season"].map(
        lambda s: season_months.get(s, (None, None))[1]
    )
    return df


def add_month_range(start_month, end_month):
    if start_month | end_month == None:
        return []
    elif start_month <= end_month:
        return list(range(start_month, end_month + 1))
    else:
        return list(range(start_month, 13)) + list(range(1, end_month + 1))


crop_yield_df = add_month(crop_yield_df)
crop_yield_df["season_months"] = crop_yield_df.apply(
    lambda row: add_month_range(row["season_start_month"], row["season_end_month"]),
    axis=1,
)

In [31]:
precipitation_df["year"] = precipitation_df["date"].dt.year
precipitation_df["month"] = precipitation_df["date"].dt.month
precipitation_df = (
    precipitation_df.groupby(["state", "district", "year", "month"], observed=True)
    .agg(precipitation_mm=("Precipitation_mm", "sum"))
    .reset_index()
)

In [32]:
vegetation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140080 entries, 0 to 140079
Data columns (total 26 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   district_id              140080 non-null  int64         
 1   district                 139536 non-null  category      
 2   state_id                 140080 non-null  int64         
 3   state                    140080 non-null  category      
 4   boundary_version         140080 non-null  object        
 5   date                     140080 non-null  datetime64[ns]
 6   period_end               140080 non-null  datetime64[ns]
 7   year                     140080 non-null  int64         
 8   month                    140080 non-null  int64         
 9   source_dataset           140080 non-null  object        
 10  source_image_count       140080 non-null  int64         
 11  ndvi_mean                80581 non-null   float64       
 12  evi_mean        

In [33]:
# drop unwanted columns from the dataset
vegetation_df.drop(
    [
        "district_id",
        "state_id",
        "boundary_version",
        "period_end",
        "source_dataset",
        "source_image_count",
        "ndvi_mean",
        "evi_mean",
        "raw_evi_mean",
        "previous_composite_date",
        "composite_gap_days",
        "ndvi_lag_1period",
        "evi_lag_1period",
        "ndvi_change",
        "ndvi_slope_per_day",
        "crop_mask_year",
    ],
    axis=1,
    inplace=True,
)

In [34]:
#  remove records with crop_area_km2 , valid_crop_area_km2 or valid_crop_fraction as NaN
vegetation_df.dropna(
    subset=["crop_area_km2", "valid_crop_area_km2", "valid_crop_fraction"], inplace=True
)
zero_crop_data = (
    vegetation_df[["crop_area_km2", "valid_crop_area_km2", "valid_crop_fraction"]] == 0
).any(axis=1)
vegetation_df = vegetation_df[~zero_crop_data]

In [35]:
vegetation_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 124189 entries, 0 to 140079
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   district             123995 non-null  category      
 1   state                124189 non-null  category      
 2   date                 124189 non-null  datetime64[ns]
 3   year                 124189 non-null  int64         
 4   month                124189 non-null  int64         
 5   raw_ndvi_mean        124189 non-null  float64       
 6   crop_area_km2        124189 non-null  float64       
 7   valid_crop_area_km2  124189 non-null  float64       
 8   valid_crop_fraction  124189 non-null  float64       
 9   quality_status       124189 non-null  category      
dtypes: category(3), datetime64[ns](1), float64(4), int64(2)
memory usage: 8.1 MB


In [36]:
#  Fill the records that has NaN for raw_ndvi_mean

vegetation_df = vegetation_df.sort_values(["state", "district", "date"])
fill_status = ["no_valid_pixels", "no_mapped_cropland", "missing_source_image"]
needs_filling = vegetation_df["raw_ndvi_mean"].isna() & vegetation_df[
    "quality_status"
].isin(fill_status)

vegetation_df.loc[needs_filling, "raw_ndvi_mean"] = vegetation_df.groupby(
    ["state", "district"], group_keys=False
)["raw_ndvi_mean"].apply(lambda x: x.bfill())[needs_filling]

still_nan = vegetation_df["raw_ndvi_mean"].isna() & needs_filling
vegetation_df.loc[still_nan, "raw_ndvi_mean"] = vegetation_df.groupby(
    ["state", "district", "date"], group_keys=False
)["raw_ndvi_mean"].apply(lambda x: x.ffill())[still_nan]

In [37]:
vegetation_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 124189 entries, 42820 to 1853
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   district             123995 non-null  category      
 1   state                124189 non-null  category      
 2   date                 124189 non-null  datetime64[ns]
 3   year                 124189 non-null  int64         
 4   month                124189 non-null  int64         
 5   raw_ndvi_mean        124189 non-null  float64       
 6   crop_area_km2        124189 non-null  float64       
 7   valid_crop_area_km2  124189 non-null  float64       
 8   valid_crop_fraction  124189 non-null  float64       
 9   quality_status       124189 non-null  category      
dtypes: category(3), datetime64[ns](1), float64(4), int64(2)
memory usage: 12.1 MB


In [38]:
precipitation_processed = project_root / "data" / "processed" / "precipitation.csv"
crop_yield_processed1 = project_root / "data" / "processed" / "crop_yield1.csv"
crop_yield_processed2 = project_root / "data" / "processed" / "crop_yield2.csv"
vegetation_processed = project_root / "data" / "processed" / "vegetation.csv"
geography_processed = project_root / "data" / "processed" / "geography.csv"

vegetation_df.to_csv(vegetation_processed)
precipitation_df.to_csv(precipitation_processed)
crop_yield_df[:200000:].to_csv(crop_yield_processed1)
crop_yield_df[200000:].to_csv(crop_yield_processed2)
geography_df.to_csv(geography_processed)


precipitation_processed_pkl = project_root / "data" / "processed" / "precipitation.pkl"
crop_yield_processed_pkl1 = project_root / "data" / "processed" / "crop_yield1.pkl"
crop_yield_processed_pkl2 = project_root / "data" / "processed" / "crop_yield2.pkl"
vegetation_processed_pkl = project_root / "data" / "processed" / "vegetation.pkl"
geography_processed_pkl = project_root / "data" / "processed" / "geography.pkl"

vegetation_df.to_pickle(vegetation_processed_pkl)
precipitation_df.to_pickle(precipitation_processed_pkl)
crop_yield_df[:200000].to_pickle(crop_yield_processed_pkl1)
crop_yield_df[200000:].to_pickle(crop_yield_processed_pkl2)
geography_df.to_pickle(geography_processed_pkl)